<!-- NB_DOC_INTRO_V1 -->
# ML — Multi-output regression & multi-label classification

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Multi-output** : predire plusieurs cibles **simultanement**. Deux cas :
- **Multi-output regression** : N variables continues (ex: predire prix + surface)
- **Multi-label classification** : 1 input → plusieurs labels possibles (ex: tags d'un article)

Couvre : `MultiOutputRegressor/Classifier`, `ClassifierChain` (utilise les predictions precedentes), modeles qui supportent natif (RF, kNN), DL (1 reseau N sorties).

## Plan

1. Multi-output regression (3 cibles continues)
2. MultiOutputRegressor (wrapper sklearn)
3. Modeles natifs (RandomForestRegressor)
4. Metriques multi-output
5. Multi-label classification
6. MultiOutputClassifier
7. ClassifierChain (chained predictions)
8. Metriques multi-label (Hamming, Jaccard, micro/macro F1)
9. Pieges et anti-patterns
10. References


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression, make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier, ClassifierChain
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score, root_mean_squared_error,
    hamming_loss, jaccard_score, f1_score, accuracy_score,
)
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Multi-output regression — generation

In [2]:
# 3 cibles continues correleees aux features
X, Y = make_regression(n_samples=1000, n_features=10, n_targets=3, noise=20, random_state=SEED)
print(f"X shape : {X.shape}, Y shape : {Y.shape}")
print(f"Y stats par cible :")
print(pd.DataFrame(Y, columns=["y0", "y1", "y2"]).describe().T[["mean", "std"]])

X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.25, random_state=SEED)

X shape : (1000, 10), Y shape : (1000, 3)
Y stats par cible :
        mean         std
y0 -3.184379  161.173437
y1 -4.688578  187.830052
y2 -1.347354  182.764047


## 2. `MultiOutputRegressor` — wrapper (1 modele par cible)

In [3]:
# Wrap n'importe quel regressor mono-cible
mor = MultiOutputRegressor(GradientBoostingRegressor(random_state=SEED), n_jobs=-1)
mor.fit(X_tr, Y_tr)
Y_pred = mor.predict(X_te)

print(f"Y_pred shape : {Y_pred.shape}")
print(f"RMSE par cible : {root_mean_squared_error(Y_te, Y_pred, multioutput='raw_values').round(2)}")
print(f"R² par cible : {r2_score(Y_te, Y_pred, multioutput='raw_values').round(3)}")
print(f"R² uniform avg : {r2_score(Y_te, Y_pred, multioutput='uniform_average'):.3f}")

Y_pred shape : (250, 3)
RMSE par cible : [58.04 61.81 63.14]
R² par cible : [0.878 0.893 0.892]
R² uniform avg : 0.888


## 3. Modeles avec support natif multi-output

**RandomForest, kNN, MLP, LinearRegression** supportent natif (sans wrapper). Plus efficace que `MultiOutputRegressor` car ils peuvent partager des splits/structure entre cibles.

In [4]:
rf_native = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_native.fit(X_tr, Y_tr)
Y_pred_rf = rf_native.predict(X_te)
print(f"RF natif R² par cible : {r2_score(Y_te, Y_pred_rf, multioutput='raw_values').round(3)}")

RF natif R² par cible : [0.711 0.714 0.717]


## 4. Metriques multi-output

| Strategie | Sens |
|---|---|
| `multioutput='raw_values'` | Renvoie 1 valeur par cible (array) |
| `multioutput='uniform_average'` | Moyenne non-ponderee |
| `multioutput='variance_weighted'` | Ponderee par variance (cibles importantes plus de poids) |


## 5. Multi-label classification

In [5]:
X, Y_ml = make_multilabel_classification(n_samples=1000, n_features=20, n_classes=5,
                                          n_labels=2, random_state=SEED)
print(f"X : {X.shape}, Y : {Y_ml.shape}")
print(f"Distribution nb labels par sample :\n{pd.Series(Y_ml.sum(axis=1)).value_counts().sort_index()}")

X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y_ml, test_size=0.25, random_state=SEED)

X : (1000, 20), Y : (1000, 5)
Distribution nb labels par sample :
0    127
1    318
2    261
3    179
4     82
5     33
Name: count, dtype: int64


## 6. `MultiOutputClassifier` — 1 binaire par label

In [6]:
moc = MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=SEED), n_jobs=-1)
moc.fit(X_tr, Y_tr)
Y_pred_ml = moc.predict(X_te)

# Metriques specifiques multi-label
print(f"Hamming loss     : {hamming_loss(Y_te, Y_pred_ml):.4f}  (fraction de labels mal predits)")
print(f"Subset accuracy  : {accuracy_score(Y_te, Y_pred_ml):.4f}  (TOUS les labels exacts — severe)")
print(f"Jaccard (samples): {jaccard_score(Y_te, Y_pred_ml, average='samples'):.4f}")
print(f"F1 (samples)     : {f1_score(Y_te, Y_pred_ml, average='samples'):.4f}")
print(f"F1 (micro)       : {f1_score(Y_te, Y_pred_ml, average='micro'):.4f}")
print(f"F1 (macro)       : {f1_score(Y_te, Y_pred_ml, average='macro'):.4f}")

Hamming loss     : 0.1904  (fraction de labels mal predits)
Subset accuracy  : 0.4240  (TOUS les labels exacts — severe)
Jaccard (samples): 0.6240
F1 (samples)     : 0.6965
F1 (micro)       : 0.7361
F1 (macro)       : 0.7063


## 7. `ClassifierChain` — utilise les predictions precedentes en feature

Attention : ordre des labels compte (mais sklearn peut le shuffle aleatoirement).

In [7]:
cc = ClassifierChain(LogisticRegression(max_iter=1000, random_state=SEED), order="random", random_state=SEED)
cc.fit(X_tr, Y_tr)
Y_pred_cc = cc.predict(X_te)
print(f"ClassifierChain — F1 samples : {f1_score(Y_te, Y_pred_cc, average='samples'):.4f}")
print(f"ClassifierChain — Hamming    : {hamming_loss(Y_te, Y_pred_cc):.4f}")

ClassifierChain — F1 samples : 0.6725
ClassifierChain — Hamming    : 0.1952


## 8. RandomForest natif multi-label

In [8]:
rf_ml = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_ml.fit(X_tr, Y_tr)
Y_pred_rf = rf_ml.predict(X_te)
print(f"RF natif multi-label :")
print(f"  F1 samples : {f1_score(Y_te, Y_pred_rf, average='samples'):.4f}")
print(f"  Hamming    : {hamming_loss(Y_te, Y_pred_rf):.4f}")

RF natif multi-label :
  F1 samples : 0.6869
  Hamming    : 0.1920


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Confondre **multi-output** (N cibles indep) et **multi-task** (cibles correle es, learning conjoint) | Multi-task = DL avec shared layers + heads par tache |
| Optimiser `accuracy` en multi-label | Toujours severe — preferer Hamming / Jaccard / F1 samples |
| ClassifierChain sans `order` | Default = ordre alphabetique des cibles (souvent sub-optimal) |
| RF "natif" sur 100 cibles | Peut etre tres lent — wrapper en parallel |
| Reporter R² global sans desagregation | Toujours regarder par cible (variance_weighted) |
| Pas regarder la distribution de Y | Si Y0 a std=1 et Y1 a std=1000, RMSE incomparables |


## References

### Documentation
- sklearn multioutput : https://scikit-learn.org/stable/modules/multiclass.html
- DS_Metrics_Reference (section multi-label) : voir notebook

### Voir aussi
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [DL_PyTorch.ipynb](DL_PyTorch.ipynb)
